# TP Jour 1 — Stockage Distribué HDFS, Tolérance aux Pannes & Couche Bronze

Bienvenue dans ce premier volet pratique de notre architecture **Data Lakehouse** !

### Objectifs de la séance
1. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) et vérifier sa configuration (`dfs.blocksize = 16 Mo`, `dfs.replication = 2`).
2. **Récupérer le dataset Kaggle Olist Brazilian E-Commerce** dans `/workspace/data_raw/`.
3. **Ingérer les fichiers bruts sur HDFS** et observer physiquement le découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : simuler la perte d'un DataNode et observer l'auto-réparation (ré-allocation automatique des blocs sous-répliqués en ~50 secondes).
5. **Construire la couche Bronze** (`/lakehouse/bronze/`) avec PySpark et inspecter notre premier DAG sur **Spark Web UI** (`http://localhost:4040`).

> **Auto-validation :** Une fois toutes les cellules `# TODO` complétées, vous pouvez valider votre notebook depuis un terminal avec `task test:j1`.

---
### Interfaces utiles pendant le TP
| Service | URL | Identifiant |
|---|---|---|
| **HDFS NameNode UI** | http://localhost:9870 | — |
| **JupyterLab** | http://localhost:8888 | Token : `bigdata` |
| **Spark Web UI** | http://localhost:4040 | *(Actif dès que `SparkSession` démarre)* |
| **Dashboard Streamlit** | http://localhost:8501 | — |

## 1. Acquisition du dataset Olist Brazilian E-Commerce (Fourni)

Le jeu de données public **Olist** regroupe ~100 000 commandes réelles passées au Brésil entre 2016 et 2018, réparties en 9 fichiers CSV (~120 Mo au total, dont `olist_geolocation_dataset.csv` d'environ **61 Mo**, idéal pour observer le découpage en blocs HDFS de **16 Mo**).

In [ ]:
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

existing_csvs = sorted(DATA_RAW.glob("*.csv"))
if len(existing_csvs) >= 8:
    print(f"Dataset déjà présent dans {DATA_RAW} ({len(existing_csvs)} fichiers CSV).")
else:
    import kagglehub
    print("Téléchargement du dataset olistbr/brazilian-ecommerce depuis Kaggle Hub...")
    cache_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    for csv_file in sorted(cache_dir.glob("*.csv")):
        shutil.copy2(csv_file, DATA_RAW / csv_file.name)
    existing_csvs = sorted(DATA_RAW.glob("*.csv"))
    print(f"Téléchargement terminé : {len(existing_csvs)} fichiers copiés dans {DATA_RAW}.")

print("\nInventaire des fichiers locaux (avant ingestion HDFS) :")
for f in existing_csvs:
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {f.name:<42} {size_mb:>6.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark est pré-configuré pour pointer vers `hdfs://namenode:9000` avec **Delta Lake**.

### Exercice 2.1 — Initialiser la `SparkSession` et afficher la configuration HDFS
Complétez la cellule ci-dessous pour :
1. Créer une `SparkSession` nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Afficher la valeur des clés de configuration `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"` via `spark.conf.get(...)`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# TODO : Initialiser la SparkSession avec appName("J1_HDFS_Ingestion_Bronze")
spark = None

assert spark is not None, "TODO : Initialisez la variable `spark` avec SparkSession.builder..."

# TODO : Afficher fs.defaultFS, dfs.blocksize et dfs.replication depuis spark.conf.get(...)
default_fs = ...
block_size = ...
replication = ...

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

## 3. Téléversement des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante vous fournit le pont vers l'API `org.apache.hadoop.fs.FileSystem` afin de téléverser les 9 CSV locaux vers `/lakehouse/bronze/raw_csv/` sur HDFS.

In [ ]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw_csv")
fs.mkdirs(raw_hdfs_dir)

for csv_path in sorted(DATA_RAW.glob("*.csv")):
    src = PathHDFS(f"file://{csv_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw_csv/{csv_path.name}")
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

In [ ]:
# Inspection physique du découpage en blocs HDFS (16 Mo) et des réplicas (facteur 2 sur 3 DataNodes)
for status in sorted(fs.listStatus(raw_hdfs_dir), key=lambda s: s.getLen(), reverse=True):
    path = status.getPath()
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {path.getName()} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

### Questions d'observation (à vérifier sur http://localhost:9870)
1. Sur `http://localhost:9870` (*Utilities > Browse the file system*), naviguez dans `/lakehouse/bronze/raw_csv/`.
2. Combien de blocs de 16 Mo composent `olist_geolocation_dataset.csv` (~61 Mo) ? Quelle est la taille du dernier bloc ?
3. Pourquoi le facteur de réplication est-il fixé à `2` sur un cluster à `3` DataNodes pour ce TP ?

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection de panne d'un DataNode a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis un terminal :
1. Vérifiez l'état initial : `task hdfs:fsck` (ou `docker exec namenode hdfs fsck /lakehouse/bronze/raw_csv/olist_geolocation_dataset.csv -files -blocks -locations`).
2. Coupez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Surveillez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s jusqu'au passage de `datanode2` en **Dead** et la ré-allocation automatique des blocs sous-répliqués vers les 2 DataNodes survivants.

### Exercice 4.1 — Vérifier la lecture sans interruption et la *Data Locality*
Pendant que `datanode2` est éteint, lisez `/lakehouse/bronze/raw_csv/olist_geolocation_dataset.csv` avec `spark.read.csv(...)` (options `header=true`, `inferSchema=true`), comptez le nombre de lignes et affichez `df_geo_check.rdd.getNumPartitions()`.

*(Pensez ensuite à relancer `datanode2` avec `task hdfs:restore-dn2` !)*

In [ ]:
# TODO : Lire /lakehouse/bronze/raw_csv/olist_geolocation_dataset.csv avec header=true et inferSchema=true
df_geo_check = None

assert df_geo_check is not None, "TODO : Chargez le CSV HDFS dans `df_geo_check`."

print(f"Nombre total de lignes lues depuis HDFS : {df_geo_check.count():,}")
print(f"Nombre de partitions Spark créées       : {df_geo_check.rdd.getNumPartitions()}")
# Question : Que remarquez-vous entre le nombre de partitions Spark et le nombre de blocs HDFS de 16 Mo ?

---
## 5. Construction de la Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir les 9 CSV bruts en tables Delta Lake Bronze
Dans une architecture **Médaillon**, la couche **Bronze** conserve l'intégralité des données brutes mais les stocke au format transactionnel **Delta Lake** en ajoutant deux colonnes de traçabilité technique :
- `_ingested_at` : horodatage d'ingestion (`F.current_timestamp()`)
- `_source_file` : chemin du fichier source (`F.input_file_name()`)

Complétez la boucle ci-dessous pour lire chaque CSV depuis `/lakehouse/bronze/raw_csv/{csv_filename}` (avec `header=true`, `inferSchema=true`, `multiLine=true`, `escape='"'`), ajouter les 2 colonnes techniques, et sauvegarder en format `"delta"` (`mode("overwrite")`, `option("overwriteSchema", "true")`) dans `/lakehouse/bronze/{table_name}`.

In [ ]:
BRONZE_TABLES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "geolocation": "olist_geolocation_dataset.csv",
}

for table_name, csv_filename in BRONZE_TABLES.items():
    hdfs_csv_path = f"/lakehouse/bronze/raw_csv/{csv_filename}"
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    # TODO : Lire le CSV HDFS, ajouter `_ingested_at` et `_source_file`, puis écrire en Delta Lake
    df_raw = None

    assert df_raw is not None, f"TODO : Construisez et sauvegardez la table Bronze `{table_name}`."

    # TODO : Sauvegarder `df_raw` au format "delta" (mode overwrite + overwriteSchema=true) dans `bronze_delta_path`
    ...

    row_count = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] /lakehouse/bronze/{table_name:<22} -> {row_count:>9,} lignes")

In [ ]:
# Vérification finale de la couche Bronze
orders_bronze_check = spark.read.format("delta").load("/lakehouse/bronze/orders")
assert "_ingested_at" in orders_bronze_check.columns and "_source_file" in orders_bronze_check.columns, (
    "Les colonnes techniques `_ingested_at` et `_source_file` doivent être présentes !"
)
orders_bronze_check.select("order_id", "order_status", "order_purchase_timestamp", "_ingested_at").show(5, truncate=False)